## Importación de librerias y definición de funciones 

In [345]:
import pandas as pd
from pptx import Presentation
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE
from pptx.enum.chart import XL_TICK_MARK
from pptx.enum.chart import XL_LEGEND_POSITION
from pptx.util import Inches
from pptx.util import Pt
from pptx.dml.color import RGBColor 
import lxml.etree as ET

In [346]:
brand_color = RGBColor(0,174,195)

# Color pallete throughout the code, this is the standard pallete
kantar_pallete = [
    RGBColor(0,96,255), #kantar_blue_strong
    RGBColor(128,42,183), #kantar_purple_strong
    RGBColor(0,229,186), #kantar_teal_strong
    RGBColor(0,182,0), #kantar_green_strong
    RGBColor(250,0,40), #kantar_red_strong
    RGBColor(254,219,0), #kantar_yellow_strong
    RGBColor(255,80,0), #kantar_orange_strong
    RGBColor(181,115,223), #kantar_blue_mid
    RGBColor(101,159,255), #kantar_purple_mid
    RGBColor(86,255,223), #kantar_teal_mid
    RGBColor(58,255,58), #kantar_green_mid
    RGBColor(255,99,123), #kantar_red_mid
    RGBColor(254,233,101), #kantar_yellow_mid
    RGBColor(255,149,101), #kantar_orange_mid
    RGBColor(230,208,244), #kantar_blue_light
    RGBColor(204,223,255), #kantar_purple_light
    RGBColor(198,254,244), #kantar_teal_light
    RGBColor(189,255,189), #kantar_green_light
    RGBColor(255,203,211), #kantar_red_light
    RGBColor(255,247,203), #kantar_yellow_light
    RGBColor(255,220,204), #kantar_orange_light
]

kantar_pallete.insert(0, brand_color)

In [347]:
def get_slide_placeholders(slide_object):
    """
    This is a function that prints a list of the placeholdes a slide object has. Useful for mapping the available elements for edition. 
    """
    for placeholder in slide_object.placeholders:
        print(f"{placeholder.placeholder_format.idx}: {placeholder.name}")

def get_slide_shapes(slide_object):
    """
    This is a function that prints a list of the shapes a slide object has. Useful for mapping the available elements for edition. 
    """
    for shape in slide_object.shapes:
        print(f'{shape.shape_id}:{shape.name}')

def get_presentation_layouts(presentation_object):
    """
    Function that prints a list of the layouts of a presentation object. 
    """
    counter = 0
    for layout in presentation_object.slide_layouts:
        print(f'{counter}: {layout.name}')
        counter += 1

In [348]:
def check_columns_strtype(df=pd.DataFrame(), all_columns=[]):
    
    """
    Function that checks if the columns in the all_columns list are string type, otherwise it raises an error listing all the non str type columns. 
    """

    non_string_columns = {}

    for col in all_columns:
        if not pd.api.types.is_string_dtype(df[col]):
            non_string_columns[col] = df[col].dtype

    if non_string_columns:
        error_message = "The following columns are not of string type:\n"
        for col, dtype in non_string_columns.items():
            error_message += f"{col}: {dtype}\n"
        raise ValueError(error_message)

In [349]:
def get_crosstab(stacked_df=pd.DataFrame(), index_vars=[], column_vars=[], value_var='', filter_mask=None, agg_function='sum', normalize: "bool | Literal[0, 1, 'all', 'index', 'columns']" = False):
    
    """
    Extends the functionality of the pd.crosstab function. 
    It allows to prefilter the data for the crosstab with the filter mask parameter, this parameter needs a masking pd.Series that will apply as filter. 
    Also it allows to nest variables for the index and columns elements of the crosstab. 
    
    Note: for this to function the index_vars and column_vars need to be of string type. Make sure to convert them beforehand into a string format. 
    """

    # Applies filter (if applicable)
    if filter_mask is None:
        _filtered_df = stacked_df.copy()
    else:
        _filtered_df = stacked_df[filter_mask].copy()


    # Checks if all the columns are string type, othewise it raises an error
    _all_columns = []
    _all_columns.extend(column_vars)
    _all_columns.extend(index_vars)

    check_columns_strtype(_filtered_df, _all_columns)


    # Creates nested index and column variables 
    _filtered_df['agg_index_var'] = _filtered_df[index_vars].agg(' '.join, axis=1)
    _filtered_df['agg_column_var'] = _filtered_df[column_vars].agg(' '.join, axis=1)


    # Generates the crosstab with the nested index and column variables
    _crosstab = pd.crosstab(
        index=_filtered_df['agg_index_var'],
        columns=_filtered_df['agg_column_var'],
        values=_filtered_df[value_var],
        aggfunc=agg_function,
        normalize=normalize
    )

    return _crosstab

In [350]:
def category_series_grapher(
        press_object,
        layout_object,
        slide_title='title', 
        stacked_df=pd.DataFrame(), 
        category_vars=[], 
        series_vars=[], 
        value_var='', 
        iterable_vars=[], 
        filter_mask = None, 
        agg_function='sum', 
        color_pallete=kantar_pallete,
        number_format_series='#,##0',
        chart_type=XL_CHART_TYPE.COLUMN_STACKED,
        grouping_vars=[],
        use_slide_object=False,
        slide_object=None,
        pos_horizontal=Inches(0.39),
        pos_vertical=Inches(1.87),
        width=Inches(12.54),
        height=Inches(4.37),
        show_legend=True,
        show_category_axis=True,
        show_data_labels_perc = False
        ):
    
    """
    This function generates graphs slides in 3 modes of operations: 

    - Single graph in a slide -
    This is the most simple mode, this will create a new slide with a single graph on it. For this mode of operation both iterable_vars and grouping vars should be left empty. 

    - Multiple single graphs slides -
    This mode generates multiple new slides with a single graph on each. The graph will be created by looping on the unique elements of iterable_vars object. 

    - Multiple graph triplet slides - 
    This is the most complex mode, this will create up to 3 graphs in a single slide per group of every unique element in the grouping_vars object. 
    If a group would create more than 3 graphs it will create more slides as triplets of graphs.
    This mode require sboth iterable_vars and grouping_vars lists. 

    """

    # Filter the data
    if filter_mask is None:
        _filtered_df = stacked_df.copy()
    else:
        _filtered_df = stacked_df[filter_mask].copy()

    if (len(iterable_vars) == 0) & (len(grouping_vars) == 0):

        # Prepare the data

        _crosstab = get_crosstab(
            stacked_df=_filtered_df,
            index_vars=category_vars,
            column_vars=series_vars,
            value_var=value_var,
            agg_function=agg_function
        ).fillna(0)
        
        # Load data into category chart data object 
        _chart_data = CategoryChartData()

        for category in _crosstab.index.to_list():
            _chart_data.add_category(category)
        
        for serie in _crosstab.columns.to_list():
            _chart_data.add_series(
                name=serie,
                values=_crosstab[serie].to_list(),
                number_format=number_format_series
            )
        
        # If we are in multiple graph triplet slides mode this will use the slide object for the batch 
        if use_slide_object:
            _graph_slide = slide_object
        else:
            # Initialize the slide if we are creating it for the single slide single graph mode 
            _graph_slide = press_object.slides.add_slide(layout_object)

        if not use_slide_object:
            # Add title to the slide if we are creating it for the single slide single graph mode 
            _graph_slide.placeholders[0].text = slide_title

        # Initialize the graphic frame object 
        _graphic_frame = _graph_slide.shapes.add_chart(
            chart_type=chart_type,
            x=pos_horizontal,
            y=pos_vertical,
            cx=width,
            cy=height,
            chart_data=_chart_data
        )

        # Chart object for decorating
        _chart = _graphic_frame.chart 

        # Adding data lables as % by category
        if show_data_labels_perc:
            # Normalize the crosstab by categories (rows)
            _normalized_crosstab = _crosstab.div(_crosstab.sum(axis=1), axis=0).fillna(0) * 100

            # Insertar etiquetas personalizadas en el XML del gráfico
            _chart_xml = _chart.plots[0].chart._element
            for s_idx, ser in enumerate(_chart_xml.findall('.//c:ser', namespaces=_chart_xml.nsmap)):
                dLbls = ser.find('c:dLbls', namespaces=_chart_xml.nsmap)
                if dLbls is None:
                    dLbls = ET.SubElement(ser, '{http://schemas.openxmlformats.org/drawingml/2006/chart}dLbls')
                for idx, perc in enumerate(_normalized_crosstab.iloc[:, s_idx].tolist()):
                    dLbl = ET.SubElement(dLbls, '{http://schemas.openxmlformats.org/drawingml/2006/chart}dLbl')
                    idx_tag = ET.SubElement(dLbl, '{http://schemas.openxmlformats.org/drawingml/2006/chart}idx')
                    idx_tag.set('val', str(idx))
                    tx = ET.SubElement(dLbl, '{http://schemas.openxmlformats.org/drawingml/2006/chart}tx')
                    rich = ET.SubElement(tx, '{http://schemas.openxmlformats.org/drawingml/2006/chart}rich')
                    ET.SubElement(rich, '{http://schemas.openxmlformats.org/drawingml/2006/main}bodyPr')
                    ET.SubElement(rich, '{http://schemas.openxmlformats.org/drawingml/2006/main}lstStyle')
                    p = ET.SubElement(rich, '{http://schemas.openxmlformats.org/drawingml/2006/main}p')
                    r = ET.SubElement(p, '{http://schemas.openxmlformats.org/drawingml/2006/main}r')
                    ET.SubElement(r, '{http://schemas.openxmlformats.org/drawingml/2006/main}rPr')
                    t = ET.SubElement(r, '{http://schemas.openxmlformats.org/drawingml/2006/main}t')
                    t.text = f"{perc:.1f}%"
                    dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showLegendKey', val='0'))
                    dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showVal', val='0'))
                    dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showCatName', val='0'))
                    dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showSerName', val='0'))
                    dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showPercent', val='0'))
                    dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showBubbleSize', val='0'))

        # Decoration parameters 
        _chart.has_title = False # Turns off the title for the graph 
        _chart.font.name = 'Arial' # Font for graph
        _chart.font.size = Pt(9) # Font size

        # Turning off gridlines 
        _chart.value_axis.has_major_gridlines = False 
        _chart.value_axis.has_minor_gridlines = False
        _chart.category_axis.has_major_gridlines = False
        _chart.category_axis.has_minor_gridlines = False 

        # Making the value axis start at 0 always 
        _chart.value_axis.minimum_scale = 0

        # Turning on or off the category axis 
        if not show_category_axis:
            _chart.category_axis.tick_labels.font.size = Pt(1)
            _chart.category_axis.tick_labels.font.color.rgb = RGBColor(255,255,255)

        # Turining off tickmarks 
        _chart.value_axis.major_tick_mark = XL_TICK_MARK.NONE
        _chart.category_axis.major_tick_mark = XL_TICK_MARK.NONE

        # Adding legend 
        _chart.has_legend = show_legend
        if show_legend:
            _chart.legend.position = XL_LEGEND_POSITION.TOP
            _chart.legend.include_in_layout = True

        # Series color fill 
        _extended_pallete = color_pallete

        while len(_extended_pallete) < len(_crosstab.columns.to_list()):
            _extended_pallete.extend(_extended_pallete)

        for i in range(len(_crosstab.columns.to_list())):
            _chart.series[i].format.fill.solid()
            _chart.series[i].format.fill.fore_color.rgb = _extended_pallete[i]

            _chart.series[i].format.line.fill.solid()
            _chart.series[i].format.line.fill.fore_color.rgb = _extended_pallete[i]
            _chart.series[i].format.line.width = Pt(1.5)

    elif (len(iterable_vars) > 0) & (len(grouping_vars) == 0):
        
        # Checks if iterable vars columns are all str type 
        check_columns_strtype(_filtered_df, iterable_vars)

        # Creates nested iterable vars 
        _filtered_df['agg_iterable_var'] = _filtered_df[iterable_vars].agg(' '.join, axis=1)

        # Apply the graphing loop again for each iteration. Filters the filtered df by the corresponding iteration and adds as a subtitle the iteration. 
        for iteration in _filtered_df['agg_iterable_var'].unique():

            _crosstab = get_crosstab(
                stacked_df=_filtered_df,
                index_vars=category_vars,
                column_vars=series_vars,
                value_var=value_var,
                agg_function=agg_function,
                filter_mask=_filtered_df['agg_iterable_var']==iteration
            ).fillna(0)
            
            # Load data into category chart data object 
            _chart_data = CategoryChartData()

            for category in _crosstab.index.to_list():
                _chart_data.add_category(category)
            
            for serie in _crosstab.columns.to_list():
                _chart_data.add_series(
                    name=serie,
                    values=_crosstab[serie].to_list(),
                    number_format=number_format_series
                )
            
            # Initialize the slide 
            _graph_slide = press_object.slides.add_slide(layout_object)

            # Add title to the slide 
            _graph_slide.placeholders[0].text = slide_title

            # Add subtitle to the slide 
            _graph_slide.placeholders[12].text = iteration

            # Initialize the graphic frame object 
            _graphic_frame = _graph_slide.shapes.add_chart(
                chart_type=chart_type,
                x=pos_horizontal,
                y=pos_vertical,
                cx=width,
                cy=height,
                chart_data=_chart_data
            )

            # Chart object for decorating
            _chart = _graphic_frame.chart 

            # Adding data lables as % by category
            if show_data_labels_perc:
                # Normalize the crosstab by categories (rows)
                _normalized_crosstab = _crosstab.div(_crosstab.sum(axis=1), axis=0).fillna(0) * 100

                # Insertar etiquetas personalizadas en el XML del gráfico
                _chart_xml = _chart.plots[0].chart._element
                for s_idx, ser in enumerate(_chart_xml.findall('.//c:ser', namespaces=_chart_xml.nsmap)):
                    dLbls = ser.find('c:dLbls', namespaces=_chart_xml.nsmap)
                    if dLbls is None:
                        dLbls = ET.SubElement(ser, '{http://schemas.openxmlformats.org/drawingml/2006/chart}dLbls')
                    for idx, perc in enumerate(_normalized_crosstab.iloc[:, s_idx].tolist()):
                        dLbl = ET.SubElement(dLbls, '{http://schemas.openxmlformats.org/drawingml/2006/chart}dLbl')
                        idx_tag = ET.SubElement(dLbl, '{http://schemas.openxmlformats.org/drawingml/2006/chart}idx')
                        idx_tag.set('val', str(idx))
                        tx = ET.SubElement(dLbl, '{http://schemas.openxmlformats.org/drawingml/2006/chart}tx')
                        rich = ET.SubElement(tx, '{http://schemas.openxmlformats.org/drawingml/2006/chart}rich')
                        ET.SubElement(rich, '{http://schemas.openxmlformats.org/drawingml/2006/main}bodyPr')
                        ET.SubElement(rich, '{http://schemas.openxmlformats.org/drawingml/2006/main}lstStyle')
                        p = ET.SubElement(rich, '{http://schemas.openxmlformats.org/drawingml/2006/main}p')
                        r = ET.SubElement(p, '{http://schemas.openxmlformats.org/drawingml/2006/main}r')
                        ET.SubElement(r, '{http://schemas.openxmlformats.org/drawingml/2006/main}rPr')
                        t = ET.SubElement(r, '{http://schemas.openxmlformats.org/drawingml/2006/main}t')
                        t.text = f"{perc:.1f}%"
                        dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showLegendKey', val='0'))
                        dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showVal', val='0'))
                        dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showCatName', val='0'))
                        dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showSerName', val='0'))
                        dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showPercent', val='0'))
                        dLbl.append(ET.Element('{http://schemas.openxmlformats.org/drawingml/2006/chart}showBubbleSize', val='0'))


            # Decoration parameters 
            _chart.has_title = False # Turns off the title for the graph 
            _chart.font.name = 'Arial' # Font for graph
            _chart.font.size = Pt(9) # Font size

            # Turning off gridlines 
            _chart.value_axis.has_major_gridlines = False 
            _chart.value_axis.has_minor_gridlines = False
            _chart.category_axis.has_major_gridlines = False
            _chart.category_axis.has_minor_gridlines = False 

            # Making the value axis start at 0 always 
            _chart.value_axis.minimum_scale = 0

            # Turning on or off the category axis labels 
            if not show_category_axis:
                _chart.category_axis.tick_labels.font.size = Pt(1)
                _chart.category_axis.tick_labels.font.color.rgb = RGBColor(255,255,255)

            # Turining off tickmarks 
            _chart.value_axis.major_tick_mark = XL_TICK_MARK.NONE
            _chart.category_axis.major_tick_mark = XL_TICK_MARK.NONE
        
            # Adding legend 
            _chart.has_legend = show_legend

            if show_legend:
                _chart.legend.position = XL_LEGEND_POSITION.TOP
                _chart.legend.include_in_layout = True

            # Series color fill 
            _extended_pallete = color_pallete

            while len(_extended_pallete) < len(_crosstab.columns.to_list()):
                _extended_pallete.extend(_extended_pallete)

            for i in range(len(_crosstab.columns.to_list())):
                _chart.series[i].format.fill.solid()
                _chart.series[i].format.fill.fore_color.rgb = _extended_pallete[i]

                _chart.series[i].format.line.fill.solid()
                _chart.series[i].format.line.fill.fore_color.rgb = _extended_pallete[i]
                _chart.series[i].format.line.width = Pt(1.5)

    elif (len(iterable_vars) > 0) & (len(grouping_vars) > 0):

        # Checks if iterable vars columns are all str type 
        check_columns_strtype(_filtered_df, iterable_vars)
        check_columns_strtype(_filtered_df, grouping_vars)

        # Creates a variable with full grouping and iterable vars
        _group_iter_vars = []
        _group_iter_vars.extend(grouping_vars)
        _group_iter_vars.extend(iterable_vars)

        # Creates nested iterable vars 
        _filtered_df['agg_iterable_var'] = _filtered_df[iterable_vars].agg(' '.join, axis=1)
        _filtered_df['agg_grouping_var'] = _filtered_df[grouping_vars].agg(' '.join, axis=1)
        _filtered_df['agg_group_iter_var'] = _filtered_df[_group_iter_vars].agg(' '.join, axis=1)

        for _group in _filtered_df['agg_grouping_var'].unique():

            # Elements that will form the batches we need to make graphs on 
            _items = _filtered_df[_filtered_df['agg_grouping_var']==_group]['agg_group_iter_var'].unique()

            # Batches grouped in elements of 3 
            _batches = [_items[i:i+3] for i in range(0, len(_items), 3)]

            for _batch in _batches:

                # Create the slide for the batch 
                _batch_slide = press_object.slides.add_slide(layout_object)

                # Adds the group as the title 
                _batch_slide.placeholders[0].text = _group

                for i in range(len(_batch)):

                    # Conditional parameters if it is the last graph on the slide
                    if i + 1 == len(_batch):
                        _height_conditional = Inches(1.86)
                        _show_category_axis_conditional = True

                    else:
                        _height_conditional = Inches(1.43)
                        _show_category_axis_conditional = False

                    if i == 0:
                        # Creates the graph for the first position 
                        category_series_grapher(
                            press_object=press_object,
                            layout_object=layout_object,
                            stacked_df=_filtered_df,
                            category_vars=category_vars,
                            series_vars=series_vars,
                            value_var=value_var,
                            iterable_vars=[],
                            filter_mask=filter_mask & (_filtered_df['agg_group_iter_var'] == _batch[i]),
                            agg_function=agg_function,
                            color_pallete=color_pallete,
                            number_format_series=number_format_series,
                            chart_type=chart_type,
                            grouping_vars=[],
                            use_slide_object=True,
                            slide_object=_batch_slide,
                            pos_horizontal=Inches(2.16),
                            pos_vertical=Inches(1.87),
                            width=Inches(10.77),
                            height=_height_conditional,
                            show_legend=False,
                            show_category_axis=_show_category_axis_conditional
                        )

                        # Add textbox
                        _textbox1 = _batch_slide.shapes.add_textbox(
                            Inches(0.39), # Horizontal position
                            Inches(1.87), # Vertical position
                            Inches(1.6), # Width
                            Inches(0.81) # Height
                        )

                        # Decorate textbox 
                        _text_frame1 = _textbox1.text_frame
                        _text_frame1.text = _batch[i]
                        _text_frame1.word_wrap = True

                        # Text format 
                        _paragraph1 = _text_frame1.paragraphs[0]
                        _run1 = _paragraph1.runs[0]
                        _run1.font.name = 'Arial'
                        _run1.font.size = Pt(12)


                    elif i == 1:
                        # Creates the graph for the second position 
                        category_series_grapher(
                            press_object=press_object,
                            layout_object=layout_object,
                            stacked_df=_filtered_df,
                            category_vars=category_vars,
                            series_vars=series_vars,
                            value_var=value_var,
                            iterable_vars=[],
                            filter_mask=filter_mask & (_filtered_df['agg_group_iter_var'] == _batch[i]),
                            agg_function=agg_function,
                            color_pallete=color_pallete,
                            number_format_series=number_format_series,
                            chart_type=chart_type,
                            grouping_vars=[],
                            use_slide_object=True,
                            slide_object=_batch_slide,
                            pos_horizontal=Inches(2.16),
                            pos_vertical=Inches(3.32),
                            width=Inches(10.77),
                            height=_height_conditional,
                            show_legend=False,
                            show_category_axis=_show_category_axis_conditional
                        )

                        # Add textbox
                        _textbox2 = _batch_slide.shapes.add_textbox(
                            Inches(0.39), # Horizontal position
                            Inches(3.32), # Vertical position
                            Inches(1.6), # Width
                            Inches(0.81) # Height
                        )

                        # Decorate textbox 
                        _text_frame2 = _textbox2.text_frame
                        _text_frame2.text = _batch[i]
                        _text_frame2.word_wrap = True

                        # Text format 
                        _paragraph2 = _text_frame2.paragraphs[0]
                        _run2 = _paragraph2.runs[0]
                        _run2.font.name = 'Arial'
                        _run2.font.size = Pt(12)

                    elif i == 2:
                        # Creates the graph for the third position 
                        category_series_grapher(
                            press_object=press_object,
                            layout_object=layout_object,
                            stacked_df=_filtered_df,
                            category_vars=category_vars,
                            series_vars=series_vars,
                            value_var=value_var,
                            iterable_vars=[],
                            filter_mask=_filtered_df['agg_group_iter_var'] == _batch[i],
                            agg_function=agg_function,
                            color_pallete=color_pallete,
                            number_format_series=number_format_series,
                            chart_type=chart_type,
                            grouping_vars=[],
                            use_slide_object=True,
                            slide_object=_batch_slide,
                            pos_horizontal=Inches(2.16),
                            pos_vertical=Inches(4.77),
                            width=Inches(10.77),
                            height=Inches(1.86),
                            show_legend=False,
                            show_category_axis=True
                        )

                        # Add textbox
                        _textbox3 = _batch_slide.shapes.add_textbox(
                            Inches(0.39), # Horizontal position
                            Inches(4.77), # Vertical position
                            Inches(1.6), # Width
                            Inches(0.81) # Height
                        )

                        # Decorate textbox 
                        _text_frame3 = _textbox3.text_frame
                        _text_frame3.text = _batch[i]
                        _text_frame3.word_wrap = True

                        # Text format 
                        _paragraph3 = _text_frame3.paragraphs[0]
                        _run3 = _paragraph3.runs[0]
                        _run3.font.name = 'Arial'
                        _run3.font.size = Pt(12)

    else: 
        _error_message = 'Invalid configuration. grouping_var list was passed but iterable_vars list is empty'
        raise ValueError(_error_message)

## Reporte

In [351]:
# Carga de datos 
data = pd.read_pickle('Output_Stacked_Data.pkl')
data['Period_str'] = data['Period'].dt.strftime('%Y-%m')
data = data[(data['Breaks']!='FY\'21') & (data['Breaks']!='FY\'25')]

In [352]:
config = {
    'L1 Label de marca': ' Jugo Maggi',
    'L1 Label de categoría': 'Category',
    'L0 Label de ventas': 'Sales',
    'L0 Label de distribución': 'Nominal Dist.',
    'L0 Label de descuento': 'Discounts',
    'L0 Label de precio': 'Price',
    'L0 Label de macroeconómicos': 'Macroeconomic',
    'Metric Label de ventas en volumen': 'L',
    'L0 Label media organica': 'Organic media'
}


In [353]:
# Inicialización de la presentación y selección de slide layouts 
prs = Presentation('Kantar Master Template.pptx')

layout_title_white = prs.slide_layouts[0]
layout_section_header_white = prs.slide_layouts[4]
layout_content_one = prs.slide_layouts[8]
layout_content_two = prs.slide_layouts[10]
layout_blank = prs.slide_layouts[22]

In [354]:
# Slide de titulo 
title_slide = prs.slides.add_slide(layout_title_white)

title_slide.placeholders[0].text = 'Data Review Deck'
title_slide.placeholders[1].text = 'Marca'

### Sección 1: Ventas

In [355]:
header_1_slide = prs.slides.add_slide(layout_section_header_white)

header_1_slide.placeholders[15].text = 'Ventas'

In [356]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    stacked_df=data,
    slide_title='Ventas de la categoría',
    category_vars=['Breaks'],
    series_vars=['L1'],
    iterable_vars=['Metric'],
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']!=config['L1 Label de categoría']),
    value_var='value',
    chart_type=XL_CHART_TYPE.COLUMN_STACKED,
    show_data_labels_perc=True
)

In [357]:
slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Ventas de la categoría'

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L1'],
    value_var='value',
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']!=config['L1 Label de categoría']) & (data['Metric'] == '$'),
    pos_horizontal=Inches(0.39),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    show_data_labels_perc=True
)

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L1'],
    value_var='value',
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']!=config['L1 Label de categoría']) & (data['Metric'] == config['Metric Label de ventas en volumen']),
    pos_horizontal=Inches(6.78),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    show_data_labels_perc=True
)


In [358]:
slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Ventas de la marca'

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L1'],
    value_var='value',
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']==config['L1 Label de marca']) & (data['Metric'] == '$'),
    pos_horizontal=Inches(0.39),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    show_data_labels_perc=False,
    show_legend=False
)

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L1'],
    value_var='value',
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']==config['L1 Label de marca']) & (data['Metric'] == config['Metric Label de ventas en volumen']),
    pos_horizontal=Inches(6.78),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    show_data_labels_perc=False,
    show_legend=False
)

In [359]:
header_2_slide = prs.slides.add_slide(layout_section_header_white)

header_2_slide.placeholders[15].text = 'Variables de marca, categoría y macroeconómicas'

In [360]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    stacked_df=data,
    slide_title='Distribución',
    category_vars=['Period_str'],
    series_vars=['L1'],
    filter_mask=(data['L0']==config['L0 Label de distribución']),
    value_var='value',
    chart_type=XL_CHART_TYPE.LINE,
    show_legend=False,
    show_data_labels_perc=False
)

In [361]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    stacked_df=data,
    slide_title='Precio de la marca y serie de descuento ##AJUSTAR ESCALA DEL GRAFICO##',
    category_vars=['Period_str'],
    series_vars=['L0','Metric'],
    filter_mask=((data['L0']==config['L0 Label de precio']) | (data['L0']==config['L0 Label de descuento'])) & (data['L1']==config['L1 Label de marca']),
    value_var='value',
    chart_type=XL_CHART_TYPE.LINE,
    show_legend=True,
    show_data_labels_perc=False
)

In [362]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    stacked_df=data,
    slide_title='Precio',
    category_vars=['Period_str'],
    series_vars=['L1'],
    filter_mask=(data['L0']==config['L0 Label de precio']),
    value_var='value',
    chart_type=XL_CHART_TYPE.LINE,
    show_legend=True,
    show_data_labels_perc=False
)

In [363]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    slide_title='Macroeconómicos',
    stacked_df=data,
    category_vars=['Period_str'],
    series_vars=['Metric'],
    value_var='value',
    iterable_vars=['L2','L3','L4'],
    filter_mask=(data['L0']==config['L0 Label de macroeconómicos']),
    chart_type=XL_CHART_TYPE.LINE
)

### Sección 2: Medios pagados

In [364]:
header_3_slide = prs.slides.add_slide(layout_section_header_white)

header_3_slide.placeholders[15].text = 'Medios pagados'

In [365]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    slide_title='Inversión en medios pagados',
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L2'],
    value_var='value',
    filter_mask=(data['L0']=='Media') & (data['Metric Type']=='Spend'),
    show_data_labels_perc=True
)

In [366]:
slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Inversion en medios vs ventas en volumen'

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    slide_title='Inversión en medios pagados',
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L2'],
    value_var='value',
    filter_mask=(data['L0']=='Media') & (data['Metric Type']=='Spend'),
    show_data_labels_perc=True,
    pos_horizontal=Inches(0.39),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37)
)

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Breaks'],
    series_vars=['L1'],
    value_var='value',
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']==config['L1 Label de marca']) & (data['Metric'] == config['Metric Label de ventas en volumen']),
    pos_horizontal=Inches(6.78),
    pos_vertical=Inches(1.87),
    width=Inches(6.15),
    height=Inches(4.37),
    show_data_labels_perc=False,
    show_legend=False
)

In [367]:
slide_temp = prs.slides.add_slide(layout_blank)

slide_temp.placeholders[0].text = 'Inversion en medios vs ventas en volumen ### GRAFICO DE VENTAS ENCIMA: PIMPEAR ###'

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Period_str'],
    series_vars=['L2'],
    value_var='value',
    filter_mask=(data['L0']=='Media') & (data['Metric Type']=='Spend'),
)

category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    use_slide_object=True,
    slide_object=slide_temp,
    stacked_df=data,
    category_vars=['Period_str'],
    series_vars=['L1'],
    value_var='value',
    filter_mask=(data['L0']==config['L0 Label de ventas']) & (data['L1']==config['L1 Label de marca']) & (data['Metric'] == config['Metric Label de ventas en volumen']),
    show_data_labels_perc=False,
    show_legend=False,
    chart_type=XL_CHART_TYPE.LINE
)

In [368]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    stacked_df=data,
    category_vars=['Period_str'],
    series_vars=['Metric'],
    value_var='value',
    iterable_vars=['Metric Type'],
    grouping_vars=['L2','L3','L4'],
    filter_mask=(data['L0']=='Media'),
    chart_type=XL_CHART_TYPE.COLUMN_CLUSTERED
)

C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData

In [369]:
category_series_grapher(
    press_object=prs,
    layout_object=layout_blank,
    stacked_df=data,
    category_vars=['Period_str'],
    series_vars=['Metric'],
    value_var='value',
    iterable_vars=['Metric'],
    grouping_vars=['L2','L3','L4'],
    filter_mask=(data['L0']==config['L0 Label media organica']),
    chart_type=XL_CHART_TYPE.COLUMN_CLUSTERED
)

C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData\Local\Temp\ipykernel_26576\1988904660.py:47: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  _filtered_df = stacked_df[filter_mask].copy()
C:\Users\MonrroyF\AppData

In [370]:
t = data.drop(columns=['Period','Breaks','value','Period_str']).drop_duplicates()

## Output

In [371]:
prs.save('Preanalysis Report Output.pptx')